# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** In Croissant, record sets, fields, and columns are identified by their `@id` fields for precise referencing.

In [ ]:
# List available record sets with their '@id' and fields
print("Available record sets and their fields:\n")

if not hasattr(metadata, 'record_sets'):
    print("No record sets found in this dataset.")
else:
    for rs in metadata.record_sets:
        print(f"Record set: {rs['@id']}")
        if 'field' in rs:
            print("  Fields:")
            rs_fields = rs['field']
            if isinstance(rs_fields, dict):
                rs_fields = [rs_fields]
            for field in rs_fields:
                print(f"    - {field['@id']} (name: {field['name'] if 'name' in field else 'N/A'})")
        else:
            print("  No fields found.")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, we first determine the available record sets and pick one to demonstrate extraction.
# Let's load all record sets into pandas DataFrames.

record_set_ids = [rs['@id'] for rs in getattr(metadata, 'record_sets', [])]
dataframes = {}
for r_id in record_set_ids:
    records = list(dataset.records(record_set=r_id))
    if records:
        dataframes[r_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set '{r_id}' with columns:", dataframes[r_id].columns.tolist())
        print(dataframes[r_id].head(2))
    else:
        print(f"No data found for record set '{r_id}'.")

# For demonstration, select the first populated record set
main_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rid
        print(f"Using record set '@id': {main_record_set_id}\n")
        break

# Show the columns in the main record set
if main_record_set_id is not None:
    print("Columns for selected record set:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No populated record set found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section applies operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Below, fields are referenced via their `@id` as dictated by the Croissant schema.

In [ ]:
import numpy as np

# Ensure we have a loaded record set
if main_record_set_id is not None and not dataframes[main_record_set_id].empty:
    df = dataframes[main_record_set_id]
    # List available numeric columns
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    print('Numeric columns detected:', numeric_cols)
    
    # Select a numeric field to analyze (by column name, which should correspond to a field '@id')
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field '@id': {numeric_field_id}")

        # Filter by a threshold
        threshold = df[numeric_field_id].quantile(0.75) if not df[numeric_field_id].isnull().all() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head(3))

        # Normalize the field
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std if std != 0 else 0
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))

        # Attempt to group by another column (choose the first string/categorical field)
        category_cols = df.select_dtypes(include=['object']).columns.tolist()
        if category_cols:
            group_field_id = category_cols[0]
            print(f"Grouping by field '@id': {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            display(grouped_df.head())
        else:
            print("No categorical field available for grouping.")
    else:
        print("No numeric columns found for EDA.")
else:
    print('No data available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we show an example distribution for the selected numeric field and relationships with a categorical field (if detected).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and not dataframes[main_record_set_id].empty:
    df = dataframes[main_record_set_id]
    
    # Numeric histogram
    if 'numeric_field_id' in locals():
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(str(numeric_field_id))
        plt.ylabel('Count')
        plt.show()
    
    # If categorical grouping available, make boxplot
    if 'group_field_id' in locals():
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded a mass spectrometry dataset of extracellular vesicle proteins using the `mlcroissant` library. We inspected record sets, extracted tabular data by `@id`, analyzed a numeric field, filtered and normalized values, and visualized their distribution, demonstrating the power of Croissant-structured data for reproducible analytics.

*This notebook provides a template for reproducible, standards-based exploration of modern FAIR datasets. For more insights, consult the record set and field documentation at the provided Croissant schema URL.*